## 1. Import libraries

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

## 2. Load and inspect the dataset

Upload `Churn_Modelling.csv` to the Colab session before running this cell.

In [6]:
from google.colab import files

# This will prompt you to select and upload your Churn_Modelling.csv file
uploaded = files.upload()

Saving Churn_Modelling.csv to Churn_Modelling (1).csv


In [7]:
dataset = pd.read_csv("Churn_Modelling.csv")

dataset.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [8]:
dataset.shape

(10000, 14)

In [9]:
dataset.isnull().sum()

,0
RowNumber,0
CustomerId,0
Surname,0
CreditScore,0
Geography,0
Gender,0
Age,0
Tenure,0
Balance,0
NumOfProducts,0


## 3. Preprocess the data

In [10]:
dataset = dataset.dropna()

dataset = dataset.drop(
    columns=["RowNumber", "CustomerId", "Surname"],
    errors="ignore"
)

dataset.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [11]:
dataset = pd.get_dummies(
    dataset,
    columns=["Geography", "Gender"],
    drop_first=True,
    dtype=int
)

dataset.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1,0


## 4. Separate features and target

In [12]:
X = dataset.drop("Exited", axis=1)
y = dataset["Exited"]

X.head(), y.head()

(   CreditScore  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
 0          619   42       2       0.00              1          1   
 1          608   41       1   83807.86              1          0   
 2          502   42       8  159660.80              3          1   
 3          699   39       1       0.00              2          0   
 4          850   43       2  125510.82              1          1   
 
    IsActiveMember  EstimatedSalary  Geography_Germany  Geography_Spain  \
 0               1        101348.88                  0                0   
 1               1        112542.58                  0                1   
 2               0        113931.57                  0                0   
 3               0         93826.63                  0                0   
 4               1         79084.10                  0                1   
 
    Gender_Male  
 0            0  
 1            0  
 2            0  
 3            0  
 4            0  ,
 0    1
 1    0
 2    1

## 5. Split the dataset into training and testing sets

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=1,
    stratify=y
)

X_train.shape, X_test.shape

((8000, 11), (2000, 11))

## 6. Feature scaling

In [14]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 7. Build the Artificial Neural Network

In [15]:
model = Sequential()

model.add(Dense(16, activation="relu", input_shape=(X_train.shape[1],)))
model.add(Dense(8, activation="relu"))
model.add(Dense(1, activation="sigmoid"))

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 337 (1.32 KB)

 Trainable params: 337 (1.32 KB)

 Non-trainable params: 0 (0.00 B)

## 8. Compile the model

In [16]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## 9. Train the model

In [17]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

Epoch 1/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7742 - loss: 0.5333 - val_accuracy: 0.8012 - val_loss: 0.4740
Epoch 2/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8099 - loss: 0.4434 - val_accuracy: 0.8125 - val_loss: 0.4367
Epoch 3/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8176 - loss: 0.4241 - val_accuracy: 0.8175 - val_loss: 0.4234
Epoch 4/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8247 - loss: 0.4126 - val_accuracy: 0.8250 - val_loss: 0.4088
Epoch 5/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8319 - loss: 0.3999 - val_accuracy: 0.8325 - val_loss: 0.3944
Epoch 6/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8381 - loss: 0.3856 - val_accuracy: 0.8438 - val_loss: 0.3775
Epoch 7/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8474 - loss: 0.3703 - val_accuracy: 0.8575 - val_loss: 0.3625
Epoch 8/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8560 - loss: 0.3575 - val_accuracy: 0.

## 10. Predict customer churn

In [18]:
y_prob = model.predict(X_test)
y_pred = (y_prob >= 0.5).astype(int).ravel()

y_pred[:10]

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


array([0, 0, 0, 1, 1, 1, 0, 0, 0, 0])

## 11. Evaluate the model

In [19]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.8535

In [20]:
confusion_matrix(y_test, y_pred)

array([[1529,   64],
       [ 229,  178]])

In [21]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.96      0.91      1593
           1       0.74      0.44      0.55       407

    accuracy                           0.85      2000
   macro avg       0.80      0.70      0.73      2000
weighted avg       0.84      0.85      0.84      2000



## 12. Conclusion

In [22]:
if accuracy >= 0.5:
    print("ANN model training and evaluation completed.")
else:
    print("ANN model completed; consider improving preprocessing or model settings.")

ANN model training and evaluation completed.
